# Week 2 — Feature Engineering
**Date:** March 2026  
**Goal:** Build all player, venue, and matchup features that power the 6 modules  
**Input:** master_ipl.csv  
**Output:** features_players.csv, features_venues.csv, features_matchups.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the master dataset
master = pd.read_csv('../data/master_ipl.csv')
matches = pd.read_csv('../data/matches_clean.csv')

# Fix date columns
master['date'] = pd.to_datetime(master['date'])
matches['date'] = pd.to_datetime(matches['date'])

print("Data loaded!")
print(f"Master shape: {master.shape}")
print(f"Matches shape: {matches.shape}")

Data loaded!
Master shape: (278205, 32)
Matches shape: (1146, 15)


In [3]:
master['is_four'] = (master['batsman_runs'] == 4).astype(int)
master['is_six'] = (master['batsman_runs'] == 6).astype(int)

print("Columns added!")
print(f"Total fours: {master['is_four'].sum()}")
print(f"Total sixes: {master['is_six'].sum()}")

Columns added!
Total fours: 32113
Total sixes: 14353


In [4]:
batting_stats = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'batsman', 'date', 'season', 'batting_team'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls_faced=('batsman_runs', 'count'),
                     fours=('is_four', 'sum'),
                     sixes=('is_six', 'sum'),
                     dismissed=('is_wicket', 'max')
                 ).reset_index())

# Calculate strike rate
batting_stats['strike_rate'] = (batting_stats['runs'] / 
                                 batting_stats['balls_faced'] * 100).round(2)

# Sort by player and date — critical for rolling features
batting_stats = batting_stats.sort_values(['batsman', 'date']).reset_index(drop=True)

print("Per-match batting stats built!")
print(f"Shape: {batting_stats.shape}")
print(f"\nSample — Virat Kohli last 5 matches:")
print(batting_stats[batting_stats['batsman'] == 'V Kohli'].tail(5)[
    ['date', 'runs', 'balls_faced', 'strike_rate', 'dismissed']
].to_string())

Per-match batting stats built!
Shape: (17336, 11)

Sample — Virat Kohli last 5 matches:
            date  runs  balls_faced  strike_rate  dismissed
16304 2025-05-03    62           33       187.88          1
16305 2025-05-23    43           25       172.00          1
16306 2025-05-27    54           30       180.00          1
16307 2025-05-29    12           12       100.00          1
16308 2025-06-03    43           35       122.86          1


In [5]:
# Sort by date and ball so everything is in chronological order
master = master.sort_values(['date', 'matchId', 'inning', 'over', 'ball']).reset_index(drop=True)

# Get match-level batting stats per player (runs scored per match)
batting_match = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'batsman'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls_faced=('batsman_runs', 'count'),
                     fours=('is_four', 'sum'),
                     sixes=('is_six', 'sum')
                 ).reset_index())

# Calculate strike rate per match
batting_match['strike_rate'] = (batting_match['runs'] / batting_match['balls_faced'] * 100).round(2)

# Sort chronologically per player
batting_match = batting_match.sort_values(['batsman', 'date']).reset_index(drop=True)

print("Match-level batting stats created!")
print(f"Shape: {batting_match.shape}")
print(f"\nSample — Virat Kohli last 5 matches:")
print(batting_match[batting_match['batsman'] == 'V Kohli'].tail(5)[['date', 'runs', 'balls_faced', 'strike_rate']])

Match-level batting stats created!
Shape: (17633, 8)

Sample — Virat Kohli last 5 matches:
            date  runs  balls_faced  strike_rate
16586 2025-05-03    62           33       187.88
16587 2025-05-23    43           25       172.00
16588 2025-05-27    54           30       180.00
16589 2025-05-29    12           12       100.00
16590 2025-06-03    43           35       122.86


In [6]:
# Function to calculate exponential decay weighted average
def exp_decay_avg(series, decay=0.85):
    """More recent matches weighted higher. decay=0.85 means each match 
    is worth 85% of the next more recent match"""
    weights = np.array([decay ** i for i in range(len(series))])
    weights = weights[::-1]  # reverse so most recent has highest weight
    return np.average(series, weights=weights)

# Calculate rolling features for each player
# We use shift(1) to avoid data leakage (only use PAST matches to predict current)
batting_features = []

for player, group in batting_match.groupby('batsman'):
    group = group.sort_values('date').reset_index(drop=True)
    
    for i in range(len(group)):
        past = group.iloc[:i]  # only matches BEFORE current
        
        row = {
            'matchId': group.iloc[i]['matchId'],
            'batsman': player,
            'date': group.iloc[i]['date'],
            # Basic rolling stats
            'matches_played': len(past),
            'career_avg': past['runs'].mean() if len(past) > 0 else 0,
            'career_sr': past['strike_rate'].mean() if len(past) > 0 else 0,
            # Last 5 matches
            'avg_last5': past['runs'].tail(5).mean() if len(past) >= 2 else 0,
            'sr_last5': past['strike_rate'].tail(5).mean() if len(past) >= 2 else 0,
            # Last 10 matches
            'avg_last10': past['runs'].tail(10).mean() if len(past) >= 2 else 0,
            'sr_last10': past['strike_rate'].tail(10).mean() if len(past) >= 2 else 0,
            # Exponential decay form score
            'form_score': exp_decay_avg(past['runs'].values) if len(past) >= 2 else 0,
            'form_sr': exp_decay_avg(past['strike_rate'].values) if len(past) >= 2 else 0,
            # Consistency (lower std = more consistent)
            'consistency': past['runs'].std() if len(past) >= 2 else 0,
            # Peak score
            'peak_score': past['runs'].max() if len(past) > 0 else 0,
            # Boundary rate
            'boundary_rate': (past['fours'].sum() + past['sixes'].sum()) / past['balls_faced'].sum() if len(past) > 0 and past['balls_faced'].sum() > 0 else 0,
        }
        batting_features.append(row)

batting_features_df = pd.DataFrame(batting_features)

print("Batting features created!")
print(f"Shape: {batting_features_df.shape}")
print(f"\nSample features for V Kohli:")
print(batting_features_df[batting_features_df['batsman'] == 'V Kohli'].tail(3)[
    ['date', 'matches_played', 'career_avg', 'avg_last5', 'form_score', 'consistency']
].to_string())

Batting features created!
Shape: (17633, 15)

Sample features for V Kohli:
            date  matches_played  career_avg  avg_last5  form_score  consistency
16588 2025-05-27             256   33.445312       59.8   48.821026    27.446864
16589 2025-05-29             257   33.525292       56.0   49.597872    27.423194
16590 2025-06-03             258   33.441860       44.4   43.958191    27.402578


In [8]:
# How does each batsman perform in each phase?
phase_batting = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'batsman', 'phase'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls=('batsman_runs', 'count')
                 ).reset_index())

phase_batting['sr'] = (phase_batting['runs'] / phase_batting['balls'] * 100).round(2)

# Pivot so each phase becomes a column
phase_pivot = phase_batting.groupby(['batsman', 'phase']).agg(
    avg_runs=('runs', 'mean'),
    avg_sr=('sr', 'mean')
).round(2).unstack(level='phase')

# Flatten column names
phase_pivot.columns = ['_'.join(col).strip() for col in phase_pivot.columns]
phase_pivot = phase_pivot.reset_index()
phase_pivot.columns = [col.replace(' ', '_') for col in phase_pivot.columns]

print("Phase batting features created!")
print(f"Shape: {phase_pivot.shape}")
print(f"Columns: {list(phase_pivot.columns)}")
print(f"\nTop 5 Death over batsmen (avg runs):")
if 'avg_runs_Death' in phase_pivot.columns:
    print(phase_pivot.nlargest(5, 'avg_runs_Death')[['batsman', 'avg_runs_Death', 'avg_sr_Death']].to_string())

Phase batting features created!
Shape: (704, 7)
Columns: ['batsman', 'avg_runs_Death', 'avg_runs_Middle', 'avg_runs_Powerplay', 'avg_sr_Death', 'avg_sr_Middle', 'avg_sr_Powerplay']

Top 5 Death over batsmen (avg runs):
           batsman  avg_runs_Death  avg_sr_Death
397  Misbah-ul-Haq           40.00        266.67
219        HM Amla           27.00        210.00
448    PC Valthaty           27.00        207.69
306      KS Bharat           26.00        200.00
111      BJ Rohrer           24.33        268.88


In [9]:
# Match-level bowling stats per bowler
bowling_match = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'bowler'])
                 .agg(
                     wickets=('is_wicket', 'sum'),
                     runs_conceded=('total_runs', 'sum'),
                     balls_bowled=('total_runs', 'count'),
                     dot_balls=('total_runs', lambda x: (x == 0).sum())
                 ).reset_index())

# Calculate economy and dot ball %
bowling_match['overs_bowled'] = bowling_match['balls_bowled'] / 6
bowling_match['economy'] = (bowling_match['runs_conceded'] / bowling_match['overs_bowled']).round(2)
bowling_match['dot_pct'] = (bowling_match['dot_balls'] / bowling_match['balls_bowled'] * 100).round(2)

# Sort chronologically
bowling_match = bowling_match.sort_values(['bowler', 'date']).reset_index(drop=True)

# Rolling bowling features
bowling_features = []

for player, group in bowling_match.groupby('bowler'):
    group = group.sort_values('date').reset_index(drop=True)
    
    for i in range(len(group)):
        past = group.iloc[:i]
        
        row = {
            'matchId': group.iloc[i]['matchId'],
            'bowler': player,
            'date': group.iloc[i]['date'],
            'career_wickets': past['wickets'].sum() if len(past) > 0 else 0,
            'career_economy': past['economy'].mean() if len(past) > 0 else 0,
            'economy_last5': past['economy'].tail(5).mean() if len(past) >= 2 else 0,
            'wickets_last5': past['wickets'].tail(5).sum() if len(past) >= 2 else 0,
            'economy_last10': past['economy'].tail(10).mean() if len(past) >= 2 else 0,
            'wickets_last10': past['wickets'].tail(10).sum() if len(past) >= 2 else 0,
            'dot_pct_career': past['dot_pct'].mean() if len(past) > 0 else 0,
            'dot_pct_last5': past['dot_pct'].tail(5).mean() if len(past) >= 2 else 0,
            'form_economy': exp_decay_avg(past['economy'].values) if len(past) >= 2 else 0,
            'form_wickets': exp_decay_avg(past['wickets'].values) if len(past) >= 2 else 0,
        }
        bowling_features.append(row)

bowling_features_df = pd.DataFrame(bowling_features)

print("Bowling features created!")
print(f"Shape: {bowling_features_df.shape}")
print(f"\nSample — JJ Bumrah last 3 matches:")
print(bowling_features_df[bowling_features_df['bowler'] == 'JJ Bumrah'].tail(3)[
    ['date', 'career_wickets', 'career_economy', 'economy_last5', 'wickets_last5', 'dot_pct_last5']
].to_string())

Bowling features created!
Shape: (13845, 13)

Sample — JJ Bumrah last 3 matches:
           date  career_wickets  career_economy  economy_last5  wickets_last5  dot_pct_last5
5006 2025-05-26             201        7.152324           5.57             12         54.498
5007 2025-05-30             202        7.144266           4.82             12         59.498
5008 2025-06-01             203        7.141528           5.12              9         56.166
